# Setup & Configuration

In [1]:
# ─── Standard Library ───
import os
import re
import json
import time
import base64
from io import BytesIO
from pathlib import Path

# ─── Third-Party ───
import requests
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from PIL import Image
from openai import OpenAI

### API Keys & Configurations

In [ ]:
# ─── OpenRouter credentials ───
# ─── Load credentials from .env ───
def load_env_key(key_name, default=None):
    env_path = Path(".env")
    if env_path.exists():
        with open(env_path, "r") as f:
            for line in f:
                if line.strip().startswith(f"{key_name}="):
                    return line.split("=", 1)[1].strip().strip('"').strip("'")
    return os.environ.get(key_name, default)

OPENROUTER_API_KEY = load_env_key("OPENROUTER_API_KEY", "YOUR_API_KEY_HERE")

MODEL_NAME = "openrouter/stepfun/step-3.5-flash:free"

# ─── Retry budget for every API call ───
MAX_RETRIES = 3

# ─── Dataset Paths ───
CHARTQA_PATH    = Path("../DChartQA/chartqa.parquet")
MATHVISION_PATH = Path("../DMathVision/mathvision.parquet")
SCREENQA_PATH   = Path("../DScreenQA/screenqa.parquet")
TURTLEBENCH_PATH = Path("../DTurtleBench")
MTVQA_PATH      = Path("../DMTVQA/data/mtvqa_AR.parquet")
PEARL_PATH      = Path("../DPEARL/data/pearl.parquet")

# MathVision Evaluation

### Setup

In [6]:
# ─── Load MathVision dataset ───
mathvision_df = None
if MATHVISION_PATH.exists():
    mathvision_df = pd.read_parquet(MATHVISION_PATH)
    print(f"✅ Loaded {len(mathvision_df)} rows.")
else:
    print(f"❌ Dataset not found at {MATHVISION_PATH}.")

✅ Loaded 3040 rows.


### Prompt & Evaluator

In [7]:
MATHVISION_SYSTEM_PROMPT = """
You are an expert mathematician and visual analyst. Answer the user's question using only the provided image and text.

CRITICAL INSTRUCTIONS FOR AUTOMATED EVALUATION:
1. Output ONLY the final answer. Provide zero explanations, reasoning, or conversational filler (e.g., never write "The answer is" or "Based on the image").
2. If the answer is a letter matching a multiple choice option (A, B, C, D), simply output the letter.
3. Output equations or numbers directly.
"""


def evaluate_math(ans, gt):
    """Check whether the model answer matches the ground truth (exact or substring)."""
    ans_clean = str(ans).lower().replace(" ", "")
    gt_clean = str(gt).lower().replace(" ", "")
    return ans_clean == gt_clean or gt_clean in ans_clean

### OpenRouter

In [ ]:
# ─── MathVision run variables (OpenRouter) ───
MATHVISION_OR_LIMIT = 1  # Set to None for full evaluation

In [ ]:
def run_mathvision_openrouter(df, limit=None):
    """Run MathVision evaluation through OpenRouter API."""
    print(f"🚀 Starting MathVision OpenRouter inference — {MODEL_NAME}...")

    results_dir = Path("MathVisionResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"mathvision_{safe_model_filename(MODEL_NAME)}_results.csv"
    processed, file_exists = load_progress(output_csv, ["id"])

    count = 0
    for _, row in df.iterrows():
        if limit and count >= limit:
            break
        task_id = str(row["id"])
        if task_id in processed:
            continue

        level   = str(row.get("level", "N/A"))
        subject = str(row.get("subject", "N/A"))
        query   = str(row["question"])
        gt      = str(row["answer"]).strip()

        print(f"\nTask {task_id} | {subject} | Lvl {level}")

        img_bytes = extract_image_bytes(row)
        b64 = encode_raw_bytes_to_base64(img_bytes) if img_bytes else None
        if not b64:
            print("   ⚠️ Skipping — image error.")
            continue

        prompt = f"{query}\n{MATHVISION_SYSTEM_PROMPT}"
        answer, dur = query_openrouter(prompt, b64)
        passed = evaluate_math(answer, gt)

        file_exists = append_result({
            "id": task_id, "level": level, "subject": subject,
            "image": str(row.get("image", "N/A")), "question": query,
            "ground_truth": gt, "model_answer": answer,
            "evaluation_passed": passed, "run_time": round(dur, 3),
        }, output_csv, file_exists)

        processed.add(task_id)
        count += 1
        print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}'")
        time.sleep(3)


# ─── Execute ───
if mathvision_df is not None:
    run_mathvision_openrouter(mathvision_df, limit=MATHVISION_OR_LIMIT)
else:
    print("❌ mathvision_df not loaded — run the Setup cell first.")

# ChartQA Evaluation

### Setup

In [11]:
# ─── Load ChartQA dataset ───
chartqa_df = None
if CHARTQA_PATH.exists():
    chartqa_df = pd.read_parquet(CHARTQA_PATH)
    print(f"✅ Loaded {len(chartqa_df)} rows.")
    print(f"Columns: {list(chartqa_df.columns)}")
else:
    print(f"❌ Dataset not found at {CHARTQA_PATH}.")

✅ Loaded 32719 rows.
Columns: ['imgname', 'query', 'label', 'type', 'image']


### Prompt & Evaluator

In [12]:
CHARTQA_SYSTEM_PROMPT = """
You are a strict data extraction API analyzing a chart. Provide the exact answer to the user's question using only the visual data.

STRICT FORMATTING RULES:
1. RAW DATA ONLY: Output the final answer and nothing else. Zero conversational text, zero reasoning, and zero introductory filler (Never write "The answer is" or "Based on the chart").
2. NUMBERS: Output the exact digits. Include units, currencies, or percentages only if they are the direct answer. Do not spell out numbers as words.
3. BOOLEANS: If the question implies a true/false or yes/no answer, output strictly "Yes" or "No".
4. LISTS: If the answer requires multiple chart labels, separate them strictly with a comma and a space (e.g., "France, Germany, Italy").
5. EXACT MATCH: Copy axis labels, legend items, or categories exactly as they appear in the chart image.
"""


def evaluate_chartqa(ans, gt):
    """Relaxed match: any ground-truth label found inside the model answer."""
    ans_clean = str(ans).lower().strip()
    valid_labels = gt if isinstance(gt, list) else [gt]
    return any(str(label).lower() in ans_clean for label in valid_labels)

### OpenRouter

In [ ]:
# ─── ChartQA run variables (OpenRouter) ───
CHARTQA_OR_LIMIT = 2  # Set to None for full evaluation

In [ ]:
def run_chartqa_openrouter(df, limit=None):
    """Run ChartQA evaluation through OpenRouter API."""
    print(f"🚀 ChartQA OpenRouter — {MODEL_NAME}...")

    results_dir = Path("ChartQAResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"chartqa_openrouter_{safe_model_filename(MODEL_NAME)}_results.csv"
    processed, file_exists = load_progress(output_csv, ["imgname", "query"])

    count = 0
    for index, row in df.iterrows():
        if limit and count >= limit:
            print(f"🎯 Limit of {limit} reached.")
            break

        imgname  = str(row["imgname"])
        query    = str(row["query"])
        label    = row["label"]
        qa_type  = str(row["type"])
        task_key = f"{imgname}_{query}"

        if task_key in processed:
            continue

        print(f"[{index}] Image: {imgname} | Query: {query[:60]}")

        b64 = encode_image_bytes_to_base64(row["image"])
        if not b64:
            print("   ⚠️ Skipping — image error.")
            continue

        prompt = f"{query}\n{CHARTQA_SYSTEM_PROMPT}"
        answer, dur = query_openrouter(prompt, b64)
        passed = evaluate_chartqa(answer, label)

        file_exists = append_result({
            "id": str(index), "imgname": imgname, "query": query,
            "ground_truth": str(label), "type": qa_type,
            "model_answer": answer, "evaluation_passed": passed,
            "run_time": round(dur, 3),
        }, output_csv, file_exists)

        processed.add(task_key)
        count += 1
        print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}' | GT: '{label}'")
        time.sleep(3)


# ─── Execute ───
if chartqa_df is not None:
    run_chartqa_openrouter(chartqa_df, limit=CHARTQA_OR_LIMIT)
else:
    print("❌ chartqa_df not loaded — run the Setup cell first.")

# TurtleBench Evaluation

### Prompt & Evaluator

### Dataset Loader

In [17]:
def load_turtle_bench(dataset_path):
    """Walk the DTurtleBench/Tasks folder tree and return a DataFrame of questions."""
    tasks_dir = Path(dataset_path) / "Tasks"
    if not tasks_dir.exists():
        print(f"❌ Tasks folder not found at {tasks_dir}")
        return None

    rows = []
    for task_folder in sorted(tasks_dir.iterdir()):
        if not task_folder.is_dir():
            continue
        task_id    = task_folder.name
        text_dir   = task_folder / "QA" / "text"
        code_dir   = task_folder / "QA" / "code"
        image_path = task_folder / "image" / f"{task_id}.png"

        if not text_dir.exists():
            continue
        for qfile in sorted(text_dir.glob("q*.txt")):
            qid = qfile.stem
            query_text = qfile.read_text(encoding="utf-8").strip()
            code_file  = code_dir / f"{qid}_code.txt"
            code_text  = code_file.read_text(encoding="utf-8").strip() if code_file.exists() else "No code found."

            rows.append({
                "task_id": task_id, "question_id": qid,
                "query_text": query_text, "code_text": code_text,
                "task_folder": str(task_folder), "query_path": str(qfile),
                "code_path": str(code_file), "image_path": str(image_path),
            })
    return pd.DataFrame(rows)


def find_turtlebench_path():
    """Auto-detect the DTurtleBench folder by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if parent.name.lower() == "thesis":
            candidate = parent / "DTurtleBench"
            if candidate.exists():
                return candidate
    # Absolute fallback
    fallback = Path("/Users/smeh/Desktop/Thesis/DTurtleBench")
    return fallback if fallback.exists() else None

### OpenRouter

In [ ]:
# ─── TurtleBench run variables (OpenRouter) ───
TURTLEBENCH_OR_LIMIT = 2  # Set to None for full evaluation

In [ ]:
def run_turtlebench_openrouter(df, limit=None):
    """Run TurtleBench evaluation through OpenRouter API."""
    print(f"🚀 TurtleBench OpenRouter — {MODEL_NAME}...")

    results_dir = Path("TurtleBenchResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"TurtleBench_{safe_model_filename(MODEL_NAME)}_results.csv"
    processed, file_exists = load_progress(output_csv, ["task_id", "question_id"])

    eval_df = df.head(limit) if limit else df
    for _, row in eval_df.iterrows():
        task_key = f"{row['task_id']}_{row['question_id']}"
        if task_key in processed:
            continue

        print(f"\nTask {row['task_id']} | Q: {row['question_id']}...")

        b64 = encode_image_file_to_base64(row["image_path"])
        if not b64:
            print("   ⚠️ Skipping — image error.")
            continue

        prompt = f"{row['query_text']}\n{TURTLEBENCH_SYSTEM_PROMPT}"
        model_code, gen_time = query_openrouter(prompt, b64)
        print(f"   ⏱️ Generated in {gen_time:.2f}s")
        time.sleep(2)

        gt_code = row["code_text"].strip()
        print("   🔍 Evaluating code equivalence...")
        passed = evaluate_turtle_equivalence_openrouter(model_code, gt_code)

        file_exists = append_result({
            "task_id": row["task_id"], "question_id": row["question_id"],
            "image_path": row["image_path"], "query_text": row["query_text"],
            "model_response": model_code, "ground_truth_code": gt_code,
            "codegeneration_time": gen_time, "evaluation_passed": passed,
        }, output_csv, file_exists)

        processed.add(task_key)
        print(f"   {'✅' if passed is True else '❌'} Done.")
        time.sleep(2)


# ─── Execute ───
_tb_path = find_turtlebench_path()
if _tb_path:
    _tb_df = load_turtle_bench(_tb_path)
    if _tb_df is not None and not _tb_df.empty:
        print(f"✅ Loaded {len(_tb_df)} questions.")
        run_turtlebench_openrouter(_tb_df, limit=TURTLEBENCH_OR_LIMIT)
    else:
        print("⚠️ No TurtleBench questions found.")
else:
    print("❌ DTurtleBench folder not found.")

# ScreenQA Evaluation

### Setup

In [20]:
# ─── Load ScreenQA dataset ───
screenqa_df = None
if SCREENQA_PATH.exists():
    metadata = pq.read_metadata(SCREENQA_PATH)
    print(f"📊 ScreenQA dataset has {metadata.num_rows} rows.")
    screenqa_df = pq.read_table(SCREENQA_PATH).to_pandas()
else:
    print(f"❌ Dataset not found at {SCREENQA_PATH}.")

📊 ScreenQA dataset has 86025 rows.


### Prompt & Evaluator

In [21]:
SCREENQA_SYSTEM_PROMPT = """
You are an expert visual analyst. Answer the user's question using only the provided image and text.

CRITICAL INSTRUCTIONS FOR AUTOMATED EVALUATION:
1. Output ONLY the final answer. Provide zero explanations, reasoning, or conversational filler (e.g., never write "The answer is" or "Based on the image").
"""

# ─── Image resize dimensions for ScreenQA ───
SCREENQA_RESIZE = (1280, 720)


def evaluate_screenqa(ans, gt):
    """Relaxed match with currency/symbol stripping for screen-based QA."""
    ans_str = re.sub(r'[\$\£\€\,]', '', str(ans).lower())
    gt_str  = re.sub(r'[\$\£\€\,]', '', str(gt).lower())

    ans_clean = re.sub(r'[^\w]', '', ans_str)
    gt_clean  = re.sub(r'[^\w]', '', gt_str)

    if not ans_clean or not gt_clean:
        return False
    if ans_clean == gt_clean:
        return True
    if gt_clean in ans_clean:
        return True
    # Prevent single-char false positives (e.g. "1" matching "100")
    if len(ans_clean) >= 2 and ans_clean in gt_clean:
        return True
    return False


def extract_screenqa_ground_truth(row):
    """Safely extract the ground-truth answer from ScreenQA's nested structure."""
    raw = row.get("answers", row.get("ground_truth", row.get("full_answer", [])))
    try:
        return str(raw[0]["full_answer"]).strip()
    except (IndexError, KeyError, TypeError):
        return str(raw).strip()

### OpenRouter

In [ ]:
# ─── ScreenQA run variables (OpenRouter) ───
SCREENQA_OR_LIMIT = 4  # Set to None for full evaluation

In [ ]:
def run_screenqa_openrouter(df, limit=None):
    """Run ScreenQA evaluation through OpenRouter API."""
    print(f"🚀 ScreenQA OpenRouter — {MODEL_NAME}...")

    results_dir = Path("ScreenQAResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"screenqa_openrouter_{safe_model_filename(MODEL_NAME)}_results.csv"
    processed, file_exists = load_progress(output_csv, ["id"])

    count = 0
    for _, row in df.iterrows():
        if limit and count >= limit:
            print(f"🎯 Limit of {limit} reached.")
            break

        task_id = str(row["screen_id"])
        if task_id in processed:
            continue

        image_name = str(row.get("file_name", "N/A"))
        query = str(row["question"])
        gt = extract_screenqa_ground_truth(row)

        print(f"\nTask {task_id} | Image: {image_name}")

        img_bytes = extract_image_bytes(row)
        b64 = encode_image_bytes_to_base64(img_bytes, resize=SCREENQA_RESIZE) if img_bytes else None
        if not b64:
            print("   ⚠️ Skipping — image error.")
            continue

        prompt = f"{query}\n{SCREENQA_SYSTEM_PROMPT}"
        answer, dur = query_openrouter(prompt, b64, MODEL_NAME)
        passed = evaluate_screenqa(answer, gt)

        file_exists = append_result({
            "id": task_id, "image": image_name, "question": query,
            "ground_truth": gt, "model_answer": answer,
            "evaluation_passed": passed, "run_time": round(dur, 3),
        }, output_csv, file_exists)

        processed.add(task_id)
        count += 1
        print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}'")
        time.sleep(3)


# ─── Execute ───
if screenqa_df is not None:
    run_screenqa_openrouter(screenqa_df, limit=SCREENQA_OR_LIMIT)
else:
    print("❌ screenqa_df not loaded — run the Setup cell first.")

# PEARL Evaluation

### Setup

In [24]:
# ─── Load PEARL dataset ───
pearl_df = None
if PEARL_PATH.exists():
    pearl_df = pd.read_parquet(PEARL_PATH)
    print(f"✅ Loaded {len(pearl_df)} rows.")
else:
    print(f"❌ Dataset not found at {PEARL_PATH}.")

✅ Loaded 4832 rows.


### Prompt & Evaluator

In [25]:
PEARL_SYSTEM_PROMPT = (
    "You are an expert visual analyst. Answer the user's question using only the provided image and text.\n\n"
    "CRITICAL INSTRUCTIONS FOR AUTOMATED EVALUATION:\n"
    "1. Output ONLY the final answer. Provide zero explanations, reasoning, or conversational filler "
    "(e.g., never write \"The answer is\" or \"Based on the image\").\n"
    "2. IMPORTANT: You MUST answer in the EXACT SAME LANGUAGE as the question. "
    "If the question is in Arabic, answer in Arabic. If in English, answer in English, etc.\n"
    "If it is a true and false question answer TRUE or FALSE, if it a multiple choice question give me the answer itself and NOT the letter.\n"
)


def evaluate_pearl(ans, gt):
    """Exact or substring match for PEARL answers."""
    a = str(ans).strip()
    g = str(gt).strip()
    if not a or not g:
        return False
    if a == g:
        return True
    if g in a:
        return True
    if len(a) >= 2 and a in g:
        return True
    return False

# Results Evaluation

In [28]:
# ─── Folders to scan for result CSVs ───
RESULTS_FOLDERS = [
    "MathVisionResults", "ChartQAResults", "ScreenQAResults",
    "TurtleBenchResults", "MTVQAResults", "PEARLResults",
]

In [29]:
def summarize_results(folders=RESULTS_FOLDERS):
    """Print a table of accuracy percentages for every result CSV found."""
    print(f"{'Model Name':<60} | {'Accuracy':>10}")
    print("-" * 75)

    for folder in folders:
        folder_path = Path(folder)
        if not folder_path.exists():
            continue
        for csv_file in sorted(folder_path.glob("*.csv")):
            try:
                df = pd.read_csv(csv_file)
                if "evaluation_passed" not in df.columns or len(df) == 0:
                    continue
                correct  = df["evaluation_passed"].astype(int).sum()
                accuracy = (correct / len(df)) * 100
                name = csv_file.stem.replace("_ollama", "").replace("_results", "")
                print(f"{name:<60} | {accuracy:>8.2f}%")
            except Exception as e:
                print(f"⚠️ Error processing {csv_file.name}: {e}")


summarize_results()

Model Name                                                   |   Accuracy
---------------------------------------------------------------------------
mathvision_gemma4_e4b                                        |   100.00%
mathvision_llava_7b                                          |    50.00%
mathvision_qwen3-vl_8b                                       |   100.00%
chartqa_gemma4_e4b                                           |   100.00%
chartqa_llava_7b                                             |     0.00%
chartqa_qwen3-vl_4b                                          |    75.00%
chartqa_openrouter_openrouter_healer-alpha                   |   100.00%
screenqa_gemma4_e4b                                          |   100.00%
screenqa_llava_7b                                            |     0.00%
screenqa_qwen3-vl_4b                                         |    83.33%
screenqa_openrouter_openrouter_healer-alpha                  |    75.00%
TurtleBench_openrouter_healer-alpha            